# Five-Scenario Unified Dashboard

This notebook shows all five scenarios together in one place.


In [ ]:
from pathlib import Path
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.set_theme(style='whitegrid')

def is_project_root(p: Path) -> bool:
    return (p / 'src').is_dir() and (p / 'outputs').is_dir()

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if is_project_root(p):
            return p
    for p in (Path.home() / 'Downloads').rglob('Retinal-OCT-Images'):
        if is_project_root(p):
            return p.resolve()
    raise FileNotFoundError('Cannot locate project root containing src/ and outputs/.')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('cwd =', Path.cwd())

SCENARIOS = {
    'trad_raw_official': {
        'label': 'Traditional RAW Official',
        'family': 'traditional',
        'result_candidates': [
            PROJECT_ROOT / 'outputs/tables/final_test_hog_rbf_svm.csv',
            PROJECT_ROOT / 'outputs/runs/traditional/hog/rbf_svm/test_final/seed42_img224/tables/result.csv',
        ],
    },
    'trad_raw_strict': {
        'label': 'Traditional RAW Strict',
        'family': 'traditional',
        'result_candidates': [
            PROJECT_ROOT / 'outputs/tables/final_test_hog_rbf_svm_strict_subset.csv',
            PROJECT_ROOT / 'outputs/runs/traditional/hog/rbf_svm/strict_test_subset/seed42_img224/tables/result.csv',
        ],
    },
    'trad_processed_official': {
        'label': 'Traditional Processed Official',
        'family': 'traditional',
        'result_candidates': [
            PROJECT_ROOT / 'outputs/processed_protocol/tables/final_test_hog_rbf_svm_processed.csv',
            PROJECT_ROOT / 'outputs/runs/traditional/hog/rbf_svm/test_final/seed42_img224_processed/tables/result.csv',
        ],
    },
    'dl_processed_official': {
        'label': 'DL VGG16 Processed Official',
        'family': 'deeplearning',
        'result_candidates': [
            PROJECT_ROOT / 'outputs/tables/result_dl_vgg16_test_final.csv',
            PROJECT_ROOT / 'outputs/runs/deeplearning/vgg16/test_final/seed42_img224_processed/tables/result.csv',
        ],
    },
    'dl_processed_strict': {
        'label': 'DL VGG16 Processed Strict',
        'family': 'deeplearning',
        'result_candidates': [
            PROJECT_ROOT / 'outputs/tables/final_test_dl_vgg16_strict_subset.csv',
            PROJECT_ROOT / 'outputs/runs/deeplearning/vgg16/strict_test_subset/seed42_img224_processed/tables/result.csv',
        ],
    },
}

def pick_existing(cands):
    for p in cands:
        if Path(p).exists():
            return Path(p)
    return None

rows=[]
for k, info in SCENARIOS.items():
    rp = pick_existing(info['result_candidates'])
    if rp is None:
        rows.append({'scenario':k,'label':info['label'],'family':info['family'],'status':'missing','path':''})
        continue
    df = pd.read_csv(rp)
    if df.empty:
        rows.append({'scenario':k,'label':info['label'],'family':info['family'],'status':'empty','path':str(rp)})
        continue
    row = df.iloc[0].to_dict()
    row.update({'scenario':k,'label':info['label'],'family':info['family'],'status':'ok','path':str(rp)})
    rows.append(row)

all_df = pd.DataFrame(rows)
display(all_df[['scenario','label','family','status','path']])


In [ ]:
ok = all_df[all_df['status']=='ok'].copy()
if ok.empty:
    raise RuntimeError('No scenario result files found.')

for c in ['accuracy','macro_f1','macro_precision','macro_recall']:
    ok[c] = pd.to_numeric(ok[c], errors='coerce')

show = ok.sort_values('macro_f1', ascending=False)
display(show[['label','family','accuracy','macro_f1','macro_precision','macro_recall']])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics = [('macro_f1','Macro-F1'), ('accuracy','Accuracy'), ('macro_precision','Macro-Precision'), ('macro_recall','Macro-Recall')]
for ax, (m, t) in zip(axes.flatten(), metrics):
    sns.barplot(data=show, x=m, y='label', hue='family', dodge=False, ax=ax)
    ax.set_title(t)
    ax.set_xlabel(t)
    ax.set_ylabel('')
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[:2], labels[:2], loc='lower right')

plt.tight_layout()
plt.show()


In [ ]:
# Official vs Strict deltas
pairs = [
    ('Traditional', 'trad_raw_official', 'trad_raw_strict'),
    ('DL VGG16', 'dl_processed_official', 'dl_processed_strict'),
]

def get_metric(df, scenario, metric):
    x = df[df['scenario']==scenario]
    if x.empty:
        return np.nan
    return float(x.iloc[0][metric])

rows=[]
for name, off, st in pairs:
    row={'model':name}
    for m in ['accuracy','macro_f1','macro_precision','macro_recall']:
        off_v = get_metric(ok, off, m)
        st_v  = get_metric(ok, st, m)
        row[f'{m}_official']=off_v
        row[f'{m}_strict']=st_v
        row[f'{m}_delta']=st_v-off_v if not (np.isnan(off_v) or np.isnan(st_v)) else np.nan
    rows.append(row)

delta_df = pd.DataFrame(rows)
display(delta_df)
